In [1]:
# ============================================================
# SI7-B — MRB1-Informed TRUE LinTS Attempt-Level Overlay
# ============================================================
#
# FINAL DESIGN BEING VALIDATED
# ----------------------------
# Frozen MD6 remains the base pedagogical-move selector.
#
# LinTS does NOT directly replace MD6 and does NOT update after
# every tutor turn.
#
# Instead:
#
#   start of attempt
#       ↓
#   build 12-D context
#       ↓
#   TRUE Linear Thompson Sampling selects ONE overlay arm
#       ↓
#   same arm remains fixed for the whole tutoring attempt
#       ↓
#   frozen MD6 produces move probabilities at each turn
#       ↓
#   conservative overlay may rerank only near-tied moves
#       ↓
#   Tutor Agent produces response
#       ↓
#   frozen MRB1 scores tutor response (for NEXT attempt context)
#       ↓
#   attempt ends
#       ↓
#   external evaluator + mastery_after arrive
#       ↓
#   ONE LinTS update
#
# This removes the earlier credit-assignment problem of copying
# an attempt-level evaluator reward into many turn-level bandit
# updates.
#
# ACTION SPACE
# ------------
#   baseline
#   generic_bias
#   probing_bias
#   focus_bias
#   telling_bias
#
# REWARD
# ------
#   evaluator_score / 3
#
# MRB1 is CONTEXT, not reward.
# Current-attempt MRB1 scores become available to the NEXT attempt.
#
# IMPORTANT
# ---------
# This notebook is a mechanics/interface validation only.
# Synthetic values are used ONLY for unit tests.
# It does not establish educational improvement.
# ============================================================

import copy
import inspect
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np

SEED = 42

MOVE_ORDER = [
    "generic",
    "probing",
    "focus",
    "telling",
]

ARMS = [
    "baseline",
    "generic_bias",
    "probing_bias",
    "focus_bias",
    "telling_bias",
]

BIAS_TARGET = {
    "baseline": None,
    "generic_bias": "generic",
    "probing_bias": "probing",
    "focus_bias": "focus",
    "telling_bias": "telling",
}

FEATURE_NAMES = [
    "md6_p_generic",
    "md6_p_probing",
    "md6_p_focus",
    "md6_p_telling",
    "mastery_before",
    "prev_evaluator_rate",
    "prev_mastery_delta",
    "prev_mistake_identification",
    "prev_mistake_location",
    "prev_providing_guidance",
    "prev_actionability",
    "has_previous_feedback",
]

CONTEXT_DIM = len(FEATURE_NAMES)

# Reuse the conservative gap threshold already used in the
# earlier attempt-level overlay mechanics.
GAP_THRESHOLD = 0.10

# These are provisional LinTS mechanics values.
# They are NOT yet research-tuned hyperparameters.
RIDGE_LAMBDA = 1.0
EXPLORATION_SCALE = 0.10

RESULT_DIR = Path(
    "/kaggle/working/si7b_true_lints"
)
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 78)
print("SI7-B CONFIGURATION")
print("=" * 78)
print("Context dimension:", CONTEXT_DIM)
print("Arms:", ARMS)
print("Gap threshold:", GAP_THRESHOLD)
print("Ridge lambda:", RIDGE_LAMBDA)
print("Exploration scale:", EXPLORATION_SCALE)
print("Update timing: ONE update after completed attempt")
print("Reward: evaluator_score / 3")


SI7-B CONFIGURATION
Context dimension: 12
Arms: ['baseline', 'generic_bias', 'probing_bias', 'focus_bias', 'telling_bias']
Gap threshold: 0.1
Ridge lambda: 1.0
Exploration scale: 0.1
Update timing: ONE update after completed attempt
Reward: evaluator_score / 3


In [2]:
# ============================================================
# CONTEXT + FEEDBACK CONTRACT
# ============================================================

@dataclass
class AttemptFeedbackSummary:
    evaluator_score: int
    mastery_before: float
    mastery_after: float
    mistake_identification: float
    mistake_location: float
    providing_guidance: float
    actionability: float


def unit_interval(name, value):
    value = float(value)

    if not (0.0 <= value <= 1.0):
        raise ValueError(
            f"{name} must be in [0,1], got {value}"
        )

    return value


def validate_md6_probs(probs):
    if set(probs.keys()) != set(MOVE_ORDER):
        raise ValueError(
            f"Expected exactly {MOVE_ORDER}"
        )

    arr = np.asarray(
        [float(probs[m]) for m in MOVE_ORDER],
        dtype=np.float64,
    )

    if np.any(arr < 0.0):
        raise ValueError(
            "MD6 probabilities cannot be negative."
        )

    if not np.isclose(
        arr.sum(),
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            f"MD6 probabilities sum to {arr.sum()}, not 1."
        )

    return arr


def build_context(
    initial_md6_probabilities: Dict[str, float],
    mastery_before: float,
    previous_feedback: Optional[
        AttemptFeedbackSummary
    ] = None,
):
    md6 = validate_md6_probs(
        initial_md6_probabilities
    )

    mastery_before = unit_interval(
        "mastery_before",
        mastery_before,
    )

    if previous_feedback is None:
        historical = np.zeros(
            7,
            dtype=np.float64,
        )

    else:
        if previous_feedback.evaluator_score not in {
            0, 1, 2, 3
        }:
            raise ValueError(
                "evaluator_score must be in {0,1,2,3}"
            )

        prev_before = unit_interval(
            "previous.mastery_before",
            previous_feedback.mastery_before,
        )

        prev_after = unit_interval(
            "previous.mastery_after",
            previous_feedback.mastery_after,
        )

        prev_delta = float(
            np.clip(
                prev_after - prev_before,
                -1.0,
                1.0,
            )
        )

        historical = np.asarray(
            [
                previous_feedback.evaluator_score / 3.0,
                prev_delta,
                unit_interval(
                    "previous.MI",
                    previous_feedback.mistake_identification,
                ),
                unit_interval(
                    "previous.ML",
                    previous_feedback.mistake_location,
                ),
                unit_interval(
                    "previous.PG",
                    previous_feedback.providing_guidance,
                ),
                unit_interval(
                    "previous.AC",
                    previous_feedback.actionability,
                ),
                1.0,
            ],
            dtype=np.float64,
        )

    x = np.concatenate(
        [
            md6,
            np.asarray(
                [mastery_before],
                dtype=np.float64,
            ),
            historical,
        ]
    )

    assert x.shape == (CONTEXT_DIM,)
    assert np.isfinite(x).all()

    return x


def reward_from_evaluator(
    evaluator_score: int
):
    if evaluator_score not in {
        0, 1, 2, 3
    }:
        raise ValueError(
            "evaluator_score must be in {0,1,2,3}"
        )

    return float(
        evaluator_score / 3.0
    )


print("Context/reward contract: READY")


Context/reward contract: READY


In [3]:
# ============================================================
# MRB1 ATTEMPT-LEVEL AGGREGATION
# ============================================================
#
# A tutoring attempt may contain multiple Tutor Agent responses.
# MRB1 scores each response separately.
#
# LinTS is attempt-level, so the four critic dimensions are
# summarized over the COMPLETE attempt using a simple arithmetic
# mean.
#
# No hand-crafted weighting between tutor turns is introduced.
# ============================================================

CRITIC_KEYS = [
    "mistake_identification",
    "mistake_location",
    "providing_guidance",
    "actionability",
]


def aggregate_mrb1_scores(
    per_turn_scores: List[Dict[str, float]]
):
    if not per_turn_scores:
        raise ValueError(
            "At least one MRB1-scored tutor response is required "
            "to build completed-attempt feedback."
        )

    matrix = []

    for turn_idx, scores in enumerate(
        per_turn_scores
    ):
        if set(scores.keys()) != set(CRITIC_KEYS):
            raise ValueError(
                f"Turn {turn_idx} must contain exactly "
                f"{CRITIC_KEYS}"
            )

        row = [
            unit_interval(
                f"turn{turn_idx}.{key}",
                scores[key],
            )
            for key in CRITIC_KEYS
        ]

        matrix.append(row)

    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    means = matrix.mean(axis=0)

    return {
        key: float(value)
        for key, value
        in zip(
            CRITIC_KEYS,
            means,
        )
    }


demo_scores = [
    {
        "mistake_identification": 0.80,
        "mistake_location": 0.60,
        "providing_guidance": 0.70,
        "actionability": 0.50,
    },
    {
        "mistake_identification": 1.00,
        "mistake_location": 0.80,
        "providing_guidance": 0.90,
        "actionability": 0.70,
    },
]

demo_mean = aggregate_mrb1_scores(
    demo_scores
)

assert np.isclose(
    demo_mean["mistake_identification"],
    0.90,
)
assert np.isclose(
    demo_mean["actionability"],
    0.60,
)

print("MRB1 attempt-level mean aggregation: PASSED")


MRB1 attempt-level mean aggregation: PASSED


In [4]:
# ============================================================
# CONSERVATIVE OVERLAY
# ============================================================
#
# The selected LinTS arm stays fixed for the entire attempt.
#
# baseline:
#   always use frozen MD6 argmax.
#
# *_bias:
#   target move may override MD6 ONLY when it is already close:
#
#       top_probability - target_probability <= 0.10
#
# Therefore LinTS cannot force a move that the frozen selector
# considers far less plausible.
# ============================================================

def apply_conservative_overlay(
    arm: str,
    md6_probabilities: Dict[str, float],
    gap_threshold: float = GAP_THRESHOLD,
):
    if arm not in ARMS:
        raise ValueError(
            f"Unknown arm: {arm}"
        )

    probs_arr = validate_md6_probs(
        md6_probabilities
    )

    probs = {
        move: float(value)
        for move, value
        in zip(
            MOVE_ORDER,
            probs_arr,
        )
    }

    base_move = max(
        MOVE_ORDER,
        key=lambda m: probs[m],
    )

    target = BIAS_TARGET[arm]

    if target is None:
        return {
            "base_move": base_move,
            "final_move": base_move,
            "overridden": False,
            "target_move": None,
            "probability_gap": 0.0,
        }

    gap = float(
        probs[base_move]
        - probs[target]
    )

    override = (
        target != base_move
        and gap <= gap_threshold
    )

    final_move = (
        target
        if override
        else base_move
    )

    return {
        "base_move": base_move,
        "final_move": final_move,
        "overridden": bool(override),
        "target_move": target,
        "probability_gap": gap,
    }


# Close target -> override allowed.
test_close = apply_conservative_overlay(
    "probing_bias",
    {
        "generic": 0.10,
        "probing": 0.34,
        "focus": 0.40,
        "telling": 0.16,
    },
)

assert test_close["base_move"] == "focus"
assert test_close["final_move"] == "probing"
assert test_close["overridden"] is True

# Far target -> MD6 protected.
test_far = apply_conservative_overlay(
    "probing_bias",
    {
        "generic": 0.08,
        "probing": 0.15,
        "focus": 0.67,
        "telling": 0.10,
    },
)

assert test_far["final_move"] == "focus"
assert test_far["overridden"] is False

print("Conservative overlay: PASSED")


Conservative overlay: PASSED


In [5]:
# ============================================================
# TRUE DISJOINT LINEAR THOMPSON SAMPLING
# ============================================================
#
# For each arm a:
#
#   A_a = lambda I + Σ x x^T
#   b_a = Σ r x
#
# posterior mean:
#
#   mu_a = A_a^{-1} b_a
#
# TRUE Thompson sample:
#
#   theta_tilde_a ~ Normal(
#       mu_a,
#       exploration_scale^2 * A_a^{-1}
#   )
#
# sampled score:
#
#   score_a = x^T theta_tilde_a
#
# This is posterior sampling.
# It is NOT the old UCB-like:
#
#   mean + alpha * uncertainty
# ============================================================

class TrueDisjointLinTS:
    def __init__(
        self,
        arms,
        context_dim,
        ridge_lambda=1.0,
        exploration_scale=0.10,
        seed=42,
        data_mode="synthetic",
    ):
        self.arms = list(arms)
        self.context_dim = int(
            context_dim
        )
        self.ridge_lambda = float(
            ridge_lambda
        )
        self.exploration_scale = float(
            exploration_scale
        )
        self.data_mode = str(
            data_mode
        )

        if self.data_mode not in {
            "synthetic",
            "real",
        }:
            raise ValueError(
                "data_mode must be synthetic or real"
            )

        self.rng = np.random.default_rng(
            seed
        )

        self.A = {
            arm: (
                self.ridge_lambda
                * np.eye(
                    self.context_dim,
                    dtype=np.float64,
                )
            )
            for arm in self.arms
        }

        self.b = {
            arm: np.zeros(
                self.context_dim,
                dtype=np.float64,
            )
            for arm in self.arms
        }

        self.arm_update_counts = {
            arm: 0
            for arm in self.arms
        }

        self.total_updates = 0

    def _validate_context(self, x):
        x = np.asarray(
            x,
            dtype=np.float64,
        )

        if x.shape != (
            self.context_dim,
        ):
            raise ValueError(
                f"Expected context shape {(self.context_dim,)}, "
                f"got {x.shape}"
            )

        if not np.isfinite(x).all():
            raise ValueError(
                "Context contains non-finite values."
            )

        return x

    def posterior_for_arm(
        self,
        arm,
    ):
        A_inv = np.linalg.inv(
            self.A[arm]
        )

        mean = (
            A_inv
            @ self.b[arm]
        )

        covariance = (
            self.exploration_scale ** 2
            * A_inv
        )

        return (
            mean,
            covariance,
        )

    def sample_arm_scores(
        self,
        x,
    ):
        x = self._validate_context(
            x
        )

        scores = {}
        sampled_thetas = {}

        for arm in self.arms:
            mean, covariance = (
                self.posterior_for_arm(
                    arm
                )
            )

            theta_sample = (
                self.rng.multivariate_normal(
                    mean=mean,
                    cov=covariance,
                )
            )

            sampled_thetas[arm] = (
                theta_sample
            )

            scores[arm] = float(
                x @ theta_sample
            )

        return (
            scores,
            sampled_thetas,
        )

    def select_arm(
        self,
        x,
    ):
        scores, sampled_thetas = (
            self.sample_arm_scores(
                x
            )
        )

        arm = max(
            self.arms,
            key=lambda a: scores[a],
        )

        return {
            "selected_arm": arm,
            "sampled_scores": scores,
            "sampled_thetas": {
                k: v.tolist()
                for k, v
                in sampled_thetas.items()
            },
        }

    def update(
        self,
        arm,
        x,
        reward,
    ):
        if arm not in self.arms:
            raise ValueError(
                f"Unknown arm: {arm}"
            )

        x = self._validate_context(
            x
        )

        reward = float(reward)

        if not (0.0 <= reward <= 1.0):
            raise ValueError(
                "Reward must be in [0,1]."
            )

        self.A[arm] = (
            self.A[arm]
            + np.outer(
                x,
                x,
            )
        )

        self.b[arm] = (
            self.b[arm]
            + reward * x
        )

        self.arm_update_counts[
            arm
        ] += 1

        self.total_updates += 1

    def state_dict(self):
        return {
            "arms": self.arms,
            "context_dim": self.context_dim,
            "ridge_lambda": self.ridge_lambda,
            "exploration_scale": self.exploration_scale,
            "data_mode": self.data_mode,
            "A": {
                arm: matrix.tolist()
                for arm, matrix
                in self.A.items()
            },
            "b": {
                arm: vector.tolist()
                for arm, vector
                in self.b.items()
            },
            "arm_update_counts": (
                self.arm_update_counts
            ),
            "total_updates": (
                self.total_updates
            ),
            "rng_state": copy.deepcopy(
                self.rng.bit_generator.state
            ),
        }

    def load_state_dict(
        self,
        state,
        expected_data_mode=None,
    ):
        if (
            expected_data_mode is not None
            and state["data_mode"]
            != expected_data_mode
        ):
            raise ValueError(
                "Synthetic/real state barrier triggered: "
                f"state={state['data_mode']} "
                f"expected={expected_data_mode}"
            )

        if state["arms"] != self.arms:
            raise ValueError(
                "Arm mismatch."
            )

        if int(
            state["context_dim"]
        ) != self.context_dim:
            raise ValueError(
                "Context dimension mismatch."
            )

        self.A = {
            arm: np.asarray(
                matrix,
                dtype=np.float64,
            )
            for arm, matrix
            in state["A"].items()
        }

        self.b = {
            arm: np.asarray(
                vector,
                dtype=np.float64,
            )
            for arm, vector
            in state["b"].items()
        }

        self.arm_update_counts = {
            arm: int(count)
            for arm, count
            in state[
                "arm_update_counts"
            ].items()
        }

        self.total_updates = int(
            state["total_updates"]
        )

        self.rng.bit_generator.state = (
            copy.deepcopy(
                state["rng_state"]
            )
        )


print("True LinTS class defined.")


True LinTS class defined.


In [6]:
# ============================================================
# ATTEMPT-LEVEL CONTROLLER
# ============================================================

class AttemptLevelLinTSController:
    def __init__(
        self,
        bandit,
        gap_threshold=0.10,
    ):
        self.bandit = bandit
        self.gap_threshold = float(
            gap_threshold
        )

        self.active = None
        self.previous_feedback = None

    def start_attempt(
        self,
        episode_id,
        attempt_index,
        initial_md6_probabilities,
        mastery_before,
    ):
        if self.active is not None:
            raise RuntimeError(
                "Cannot start a new attempt while one is active."
            )

        x = build_context(
            initial_md6_probabilities=(
                initial_md6_probabilities
            ),
            mastery_before=mastery_before,
            previous_feedback=(
                self.previous_feedback
            ),
        )

        selection = (
            self.bandit.select_arm(
                x
            )
        )

        self.active = {
            "episode_id": str(
                episode_id
            ),
            "attempt_index": int(
                attempt_index
            ),
            "context": x,
            "mastery_before": float(
                mastery_before
            ),
            "selected_arm": (
                selection[
                    "selected_arm"
                ]
            ),
            "sampled_scores": (
                selection[
                    "sampled_scores"
                ]
            ),
            "turns": [],
            "mrb1_turn_scores": [],
            "updates_before": (
                self.bandit.total_updates
            ),
        }

        return {
            "selected_arm": (
                self.active[
                    "selected_arm"
                ]
            ),
            "context": x.copy(),
            "sampled_scores": copy.deepcopy(
                self.active[
                    "sampled_scores"
                ]
            ),
        }

    def apply_overlay(
        self,
        turn_index,
        md6_probabilities,
    ):
        if self.active is None:
            raise RuntimeError(
                "No active attempt."
            )

        # The arm is read from the already-frozen attempt state.
        arm = self.active[
            "selected_arm"
        ]

        result = (
            apply_conservative_overlay(
                arm=arm,
                md6_probabilities=(
                    md6_probabilities
                ),
                gap_threshold=(
                    self.gap_threshold
                ),
            )
        )

        turn_record = {
            "turn_index": int(
                turn_index
            ),
            "arm": arm,
            "md6_probabilities": {
                k: float(v)
                for k, v
                in md6_probabilities.items()
            },
            **result,
        }

        self.active[
            "turns"
        ].append(
            turn_record
        )

        # Critical lifecycle guard:
        assert (
            self.bandit.total_updates
            == self.active[
                "updates_before"
            ]
        )

        return copy.deepcopy(
            turn_record
        )

    def record_mrb1_scores(
        self,
        scores,
    ):
        if self.active is None:
            raise RuntimeError(
                "No active attempt."
            )

        # Validate immediately.
        validated = {
            key: unit_interval(
                key,
                scores[key],
            )
            for key in CRITIC_KEYS
        }

        if set(scores.keys()) != set(
            CRITIC_KEYS
        ):
            raise ValueError(
                f"Expected exactly {CRITIC_KEYS}"
            )

        self.active[
            "mrb1_turn_scores"
        ].append(
            validated
        )

    def finish_attempt(
        self,
        evaluator_score,
        mastery_after,
    ):
        if self.active is None:
            raise RuntimeError(
                "No active attempt."
            )

        reward = reward_from_evaluator(
            evaluator_score
        )

        mastery_after = unit_interval(
            "mastery_after",
            mastery_after,
        )

        critic_mean = (
            aggregate_mrb1_scores(
                self.active[
                    "mrb1_turn_scores"
                ]
            )
        )

        selected_arm = (
            self.active[
                "selected_arm"
            ]
        )

        x = self.active[
            "context"
        ]

        # Exactly ONE update for the whole attempt.
        self.bandit.update(
            arm=selected_arm,
            x=x,
            reward=reward,
        )

        assert (
            self.bandit.total_updates
            == self.active[
                "updates_before"
            ] + 1
        )

        mastery_before = float(
            self.active[
                "mastery_before"
            ]
        )

        feedback = (
            AttemptFeedbackSummary(
                evaluator_score=int(
                    evaluator_score
                ),
                mastery_before=(
                    mastery_before
                ),
                mastery_after=(
                    mastery_after
                ),
                mistake_identification=(
                    critic_mean[
                        "mistake_identification"
                    ]
                ),
                mistake_location=(
                    critic_mean[
                        "mistake_location"
                    ]
                ),
                providing_guidance=(
                    critic_mean[
                        "providing_guidance"
                    ]
                ),
                actionability=(
                    critic_mean[
                        "actionability"
                    ]
                ),
            )
        )

        self.previous_feedback = (
            feedback
        )

        summary = {
            "episode_id": (
                self.active[
                    "episode_id"
                ]
            ),
            "attempt_index": (
                self.active[
                    "attempt_index"
                ]
            ),
            "selected_arm": (
                selected_arm
            ),
            "reward": reward,
            "evaluator_score": int(
                evaluator_score
            ),
            "mastery_before": (
                mastery_before
            ),
            "mastery_after": (
                mastery_after
            ),
            "mastery_delta": float(
                mastery_after
                - mastery_before
            ),
            "mrb1_attempt_mean": (
                critic_mean
            ),
            "num_turns": len(
                self.active[
                    "turns"
                ]
            ),
            "num_overrides": sum(
                int(
                    turn[
                        "overridden"
                    ]
                )
                for turn
                in self.active[
                    "turns"
                ]
            ),
            "turns": copy.deepcopy(
                self.active[
                    "turns"
                ]
            ),
            "bandit_total_updates": (
                self.bandit.total_updates
            ),
        }

        self.active = None

        return summary


print("Attempt-level controller defined.")


Attempt-level controller defined.


In [7]:
# ============================================================
# TRUE THOMPSON-SAMPLING AUDIT
# ============================================================

bandit_audit = TrueDisjointLinTS(
    arms=ARMS,
    context_dim=CONTEXT_DIM,
    ridge_lambda=RIDGE_LAMBDA,
    exploration_scale=EXPLORATION_SCALE,
    seed=SEED,
    data_mode="synthetic",
)

x_audit = build_context(
    initial_md6_probabilities={
        "generic": 0.10,
        "probing": 0.30,
        "focus": 0.45,
        "telling": 0.15,
    },
    mastery_before=0.40,
    previous_feedback=None,
)

mean0, cov0 = (
    bandit_audit.posterior_for_arm(
        "baseline"
    )
)

scores1, thetas1 = (
    bandit_audit.sample_arm_scores(
        x_audit
    )
)

scores2, thetas2 = (
    bandit_audit.sample_arm_scores(
        x_audit
    )
)

# At untouched prior, posterior mean is exactly zero.
assert np.allclose(
    mean0,
    0.0,
)

# Samples should not equal posterior mean.
assert not np.allclose(
    thetas1["baseline"],
    mean0,
)

# Two posterior-sampling calls should generate different draws.
assert not np.allclose(
    thetas1["baseline"],
    thetas2["baseline"],
)

# Verify source contains multivariate_normal, not UCB bonus logic.
source = inspect.getsource(
    TrueDisjointLinTS.sample_arm_scores
)

assert "multivariate_normal" in source
assert "sqrt(context" not in source
assert "+ uncertainty" not in source

print("Posterior mean shape:", mean0.shape)
print("Posterior covariance shape:", cov0.shape)
print("Sample score call 1:", scores1)
print("Sample score call 2:", scores2)
print("\nTRUE Thompson posterior-sampling audit: PASSED")


Posterior mean shape: (12,)
Posterior covariance shape: (12, 12)
Sample score call 1: {'baseline': -0.05831498537866523, 'generic_bias': 0.05737610800604658, 'probing_bias': 0.031097594042923445, 'focus_bias': -0.02395676492602217, 'telling_bias': -0.02699002875021357}
Sample score call 2: {'baseline': 0.017691037476097328, 'generic_bias': 0.005396632042327397, 'probing_bias': 0.0009449507957061468, 'focus_bias': -0.053863248741010325, 'telling_bias': -0.04406743155452054}

TRUE Thompson posterior-sampling audit: PASSED


In [8]:
# ============================================================
# END-TO-END SYNTHETIC LIFECYCLE TEST
# ============================================================
#
# Synthetic values here prove only software mechanics:
#   - one arm per attempt
#   - conservative reranking
#   - MRB1 aggregation
#   - exactly one post-attempt update
#   - previous outcomes enter next attempt only
# ============================================================

bandit = TrueDisjointLinTS(
    arms=ARMS,
    context_dim=CONTEXT_DIM,
    ridge_lambda=RIDGE_LAMBDA,
    exploration_scale=EXPLORATION_SCALE,
    seed=SEED,
    data_mode="synthetic",
)

controller = (
    AttemptLevelLinTSController(
        bandit=bandit,
        gap_threshold=GAP_THRESHOLD,
    )
)

# -----------------------
# ATTEMPT 0
# -----------------------

start0 = controller.start_attempt(
    episode_id="synthetic_ep_001",
    attempt_index=0,
    initial_md6_probabilities={
        "generic": 0.12,
        "probing": 0.28,
        "focus": 0.48,
        "telling": 0.12,
    },
    mastery_before=0.42,
)

arm0 = start0["selected_arm"]

print("Attempt 0 selected arm:", arm0)

updates_at_start0 = (
    bandit.total_updates
)

turn0a = controller.apply_overlay(
    turn_index=0,
    md6_probabilities={
        "generic": 0.12,
        "probing": 0.28,
        "focus": 0.48,
        "telling": 0.12,
    },
)

controller.record_mrb1_scores({
    "mistake_identification": 0.82,
    "mistake_location": 0.73,
    "providing_guidance": 0.78,
    "actionability": 0.67,
})

turn0b = controller.apply_overlay(
    turn_index=1,
    md6_probabilities={
        "generic": 0.15,
        "probing": 0.37,
        "focus": 0.39,
        "telling": 0.09,
    },
)

controller.record_mrb1_scores({
    "mistake_identification": 0.88,
    "mistake_location": 0.77,
    "providing_guidance": 0.84,
    "actionability": 0.71,
})

# Same selected arm must be used for both turns.
assert turn0a["arm"] == arm0
assert turn0b["arm"] == arm0

# No update before evaluator/mastery arrive.
assert bandit.total_updates == updates_at_start0

summary0 = controller.finish_attempt(
    evaluator_score=2,
    mastery_after=0.55,
)

assert np.isclose(
    summary0["reward"],
    2/3,
)

assert bandit.total_updates == 1

assert np.isclose(
    summary0[
        "mrb1_attempt_mean"
    ][
        "mistake_identification"
    ],
    0.85,
)

assert np.isclose(
    summary0[
        "mrb1_attempt_mean"
    ][
        "actionability"
    ],
    0.69,
)

print("\nAttempt 0 summary:")
print(
    json.dumps(
        summary0,
        indent=2,
    )
)

# -----------------------
# ATTEMPT 1 / RETEACH
# -----------------------

start1 = controller.start_attempt(
    episode_id="synthetic_ep_001",
    attempt_index=1,
    initial_md6_probabilities={
        "generic": 0.10,
        "probing": 0.34,
        "focus": 0.44,
        "telling": 0.12,
    },
    mastery_before=0.55,
)

x1 = start1["context"]

# Previous attempt feedback is now present.
assert x1[
    FEATURE_NAMES.index(
        "has_previous_feedback"
    )
] == 1.0

assert np.isclose(
    x1[
        FEATURE_NAMES.index(
            "prev_evaluator_rate"
        )
    ],
    2/3,
)

assert np.isclose(
    x1[
        FEATURE_NAMES.index(
            "prev_mastery_delta"
        )
    ],
    0.13,
)

assert np.isclose(
    x1[
        FEATURE_NAMES.index(
            "prev_mistake_identification"
        )
    ],
    0.85,
)

print("\nAttempt 1 context includes previous feedback: PASSED")

# Finish attempt 1 so lifecycle closes cleanly.
controller.apply_overlay(
    turn_index=0,
    md6_probabilities={
        "generic": 0.10,
        "probing": 0.34,
        "focus": 0.44,
        "telling": 0.12,
    },
)

controller.record_mrb1_scores({
    "mistake_identification": 0.90,
    "mistake_location": 0.82,
    "providing_guidance": 0.86,
    "actionability": 0.77,
})

summary1 = controller.finish_attempt(
    evaluator_score=3,
    mastery_after=0.68,
)

assert bandit.total_updates == 2

print("\nExactly one update per completed attempt: PASSED")
print("Total completed attempts:", 2)
print("Total LinTS updates:", bandit.total_updates)


Attempt 0 selected arm: generic_bias

Attempt 0 summary:
{
  "episode_id": "synthetic_ep_001",
  "attempt_index": 0,
  "selected_arm": "generic_bias",
  "reward": 0.6666666666666666,
  "evaluator_score": 2,
  "mastery_before": 0.42,
  "mastery_after": 0.55,
  "mastery_delta": 0.13000000000000006,
  "mrb1_attempt_mean": {
    "mistake_identification": 0.85,
    "mistake_location": 0.75,
    "providing_guidance": 0.81,
    "actionability": 0.69
  },
  "num_turns": 2,
  "num_overrides": 0,
  "turns": [
    {
      "turn_index": 0,
      "arm": "generic_bias",
      "md6_probabilities": {
        "generic": 0.12,
        "probing": 0.28,
        "focus": 0.48,
        "telling": 0.12
      },
      "base_move": "focus",
      "final_move": "focus",
      "overridden": false,
      "target_move": "generic",
      "probability_gap": 0.36
    },
    {
      "turn_index": 1,
      "arm": "generic_bias",
      "md6_probabilities": {
        "generic": 0.15,
        "probing": 0.37,
        "foc

In [9]:
# ============================================================
# PERSISTENCE + SYNTHETIC/REAL ISOLATION AUDIT
# ============================================================

persist_bandit = TrueDisjointLinTS(
    arms=ARMS,
    context_dim=CONTEXT_DIM,
    ridge_lambda=RIDGE_LAMBDA,
    exploration_scale=EXPLORATION_SCALE,
    seed=123,
    data_mode="synthetic",
)

# One update to make state nontrivial.
persist_bandit.update(
    arm="baseline",
    x=x_audit,
    reward=2/3,
)

saved_state = (
    persist_bandit.state_dict()
)

state_path = (
    RESULT_DIR
    / "si7b_synthetic_bandit_state.json"
)

with state_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        saved_state,
        f,
        indent=2,
    )

# Reload into another synthetic policy.
clone = TrueDisjointLinTS(
    arms=ARMS,
    context_dim=CONTEXT_DIM,
    ridge_lambda=RIDGE_LAMBDA,
    exploration_scale=EXPLORATION_SCALE,
    seed=999,
    data_mode="synthetic",
)

clone.load_state_dict(
    saved_state,
    expected_data_mode="synthetic",
)

assert clone.total_updates == 1
assert np.allclose(
    clone.A["baseline"],
    persist_bandit.A["baseline"],
)
assert np.allclose(
    clone.b["baseline"],
    persist_bandit.b["baseline"],
)

# RNG state persistence:
sel_original = (
    persist_bandit.select_arm(
        x_audit
    )
)
sel_clone = (
    clone.select_arm(
        x_audit
    )
)

assert (
    sel_original["selected_arm"]
    == sel_clone["selected_arm"]
)

for arm in ARMS:
    assert np.isclose(
        sel_original[
            "sampled_scores"
        ][arm],
        sel_clone[
            "sampled_scores"
        ][arm],
    )

# Synthetic -> real must be blocked.
real_policy = TrueDisjointLinTS(
    arms=ARMS,
    context_dim=CONTEXT_DIM,
    ridge_lambda=RIDGE_LAMBDA,
    exploration_scale=EXPLORATION_SCALE,
    seed=1,
    data_mode="real",
)

blocked = False

try:
    real_policy.load_state_dict(
        saved_state,
        expected_data_mode="real",
    )
except ValueError:
    blocked = True

assert blocked is True

print("JSON state persistence: PASSED")
print("RNG reproducibility after reload: PASSED")
print("Synthetic -> real state barrier: PASSED")


JSON state persistence: PASSED
RNG reproducibility after reload: PASSED
Synthetic -> real state barrier: PASSED


In [10]:
# ============================================================
# SAVE FINAL SI7-B DESIGN SPEC
# ============================================================

spec = {
    "stage": "SI7-B",
    "name": (
        "MRB1-informed true LinTS attempt-level overlay"
    ),

    "base_selector": (
        "Frozen MD6 DistilRoBERTa"
    ),

    "critic": (
        "Frozen MRB1 Epoch 4"
    ),

    "action_space": ARMS,

    "action_semantics": {
        "baseline": (
            "Use frozen MD6 argmax."
        ),
        "generic_bias": (
            "Prefer generic only when within probability gap threshold."
        ),
        "probing_bias": (
            "Prefer probing only when within probability gap threshold."
        ),
        "focus_bias": (
            "Prefer focus only when within probability gap threshold."
        ),
        "telling_bias": (
            "Prefer telling only when within probability gap threshold."
        ),
    },

    "gap_threshold": GAP_THRESHOLD,

    "context_dimension": CONTEXT_DIM,
    "context_features": FEATURE_NAMES,

    "mrb1_attempt_aggregation": (
        "Arithmetic mean of each MRB1 expected score "
        "over all Tutor Agent responses in the completed attempt."
    ),

    "update_timing": (
        "Exactly one update after a completed tutoring attempt."
    ),

    "reward": (
        "evaluator_score / 3"
    ),

    "mastery_usage": {
        "mastery_before": (
            "Current-attempt context"
        ),
        "mastery_after": (
            "Observed after attempt; used to compute previous "
            "mastery delta for next-attempt context"
        ),
        "mastery_in_primary_reward": False,
    },

    "lints": {
        "type": (
            "Disjoint true Linear Thompson Sampling"
        ),
        "posterior_mean": (
            "A_inv @ b"
        ),
        "posterior_covariance": (
            "exploration_scale^2 * A_inv"
        ),
        "selection": (
            "Sample theta independently for each arm; "
            "choose argmax(x @ theta_sample)"
        ),
        "ridge_lambda": RIDGE_LAMBDA,
        "exploration_scale": EXPLORATION_SCALE,
        "hyperparameter_status": (
            "Provisional mechanics values; require synthetic "
            "sensitivity/convergence validation before real pilot."
        ),
    },

    "safety_methodology": [
        "Frozen MD6 is never updated by LinTS.",
        "Selected overlay arm is fixed during the attempt.",
        "MRB1 scores from current attempt do not enter current decision.",
        "Evaluator/mastery outcomes do not enter current decision.",
        "No turn-level copies of attempt reward are created.",
        "Synthetic state cannot be loaded into real policy mode.",
        "Synthetic simulations support mechanics only, not educational claims.",
    ],
}

spec_path = (
    RESULT_DIR
    / "si7b_true_lints_design_spec.json"
)

with spec_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        spec,
        f,
        indent=2,
    )

print("Saved:")
print(spec_path)
print(state_path)

print("\n" + "=" * 78)
print("SI7-B VALIDATION PASSED")
print("=" * 78)
print("Algorithm: TRUE posterior-sampling LinTS")
print("Actions: 5 attempt-level conservative overlay arms")
print("Context: 12 MRB1/MD6/mastery/outcome features")
print("Reward: evaluator_score / 3")
print("Updates: exactly one per completed attempt")
print("Mid-attempt LinTS updates: BLOCKED by lifecycle")
print("Synthetic -> real state reuse: BLOCKED")


Saved:
/kaggle/working/si7b_true_lints/si7b_true_lints_design_spec.json
/kaggle/working/si7b_true_lints/si7b_synthetic_bandit_state.json

SI7-B VALIDATION PASSED
Algorithm: TRUE posterior-sampling LinTS
Actions: 5 attempt-level conservative overlay arms
Context: 12 MRB1/MD6/mastery/outcome features
Reward: evaluator_score / 3
Updates: exactly one per completed attempt
Mid-attempt LinTS updates: BLOCKED by lifecycle
Synthetic -> real state reuse: BLOCKED
